In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions

from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user", 
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message}
    messages.append(assistant_message)


def invoke_tool(response):
    tool_map = {
        "get_current_datetime": get_current_datetime,
        "add_duration_to_datetime": add_duration_to_datetime,
        "set_reminder": set_reminder,
    }
    tool_results = []
    
    for block in response.content:
        if getattr(block, "type", None) == "tool_use":
            tool_fn = tool_map.get(block.name)
            if tool_fn is None:
                raise ValueError(f"Unknown tool: {block.name}")
            result = tool_fn(**block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result),
                "is_error": False,
            })
    return tool_results


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    
    }

    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message

In [3]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [4]:
# Get current datetime

from anthropic.types import ToolParam
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Returns the current local date and time as a formatted string. "
        "Use this tool whenever the user asks what the current date or time is, "
        "needs to know today's date, or requires a timestamp for any purpose such as "
        "scheduling, logging, or date arithmetic. The output format is controlled by "
        "the 'date_format' parameter, which accepts any valid Python strftime format string "
        "(e.g. '%Y-%m-%d' for ISO dates, '%I:%M %p' for 12-hour clock time). "
        "If no format is provided, the datetime is returned in the default ISO-like format "
        "'YYYY-MM-DD HH:MM:SS'. The 'date_format' string must not be empty."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "A Python strftime format string controlling the output layout. "
                    "Common examples: '%Y-%m-%d' → '2026-05-22', "
                    "'%d/%m/%Y' → '22/05/2026', "
                    "'%I:%M %p' → '12:54 PM', "
                    "'%A, %B %d, %Y' → 'Friday, May 22, 2026'. "
                    "Defaults to '%Y-%m-%d %H:%M:%S'. Must not be an empty string."
                ),
            }
        },
        "required": []
    }
})

get_current_datetime("%Y-%m-%d %H:%M:%S")

'2026-05-22 19:07:54'

In [5]:
#Simple invokation using tools for get current datetime

messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as YYYY-MM-DD HH:MM:SS?"
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
    )

messages.append({
    "role": "assistant",
    "content": response.content,
})

print (response)
print ( response.content[0].input)
tool_result=get_current_datetime(**response.content[0].input)
print ( tool_result)


messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content" : tool_result,
        "is_error": False
    }
    ]
})

print(messages)


Message(id='msg_013SimJjs5hj4XpP3nqGmiCj', container=None, content=[ToolUseBlock(id='toolu_01WLQL2y8cGc8Mo8ggrPuYSe', caller=DirectCaller(type='direct'), input={'date_format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=851, output_tokens=69, server_tool_use=None, service_tier='standard'))
{'date_format': '%Y-%m-%d %H:%M:%S'}
2026-05-22 19:07:55
[{'role': 'user', 'content': 'What is the exact time, formatted as YYYY-MM-DD HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01WLQL2y8cGc8Mo8ggrPuYSe', caller=DirectCaller(type='direct'), input={'date_format': '%Y-%m-%d %H:%M:%S'}, name='get_current_datetime', type='to

In [6]:
#Multi turn invokation using tools for get current datetime and add duration to datetime

def run_conversation(messages):
    
    while True:
        response = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])
        #print("Response: ", response)
        #If response does not contain tooluseblock, break the loop
        if not any(getattr(block, "type", None) == "tool_use" for block in response.content):
            #print("Response does not contain tooluseblock, breaking the loop")
            return response
            break
        add_assistant_message(messages, response)
        #If response contains tooluseblock, invoke the tool and return result as user message
        tool_result = invoke_tool(response)
        add_user_message(messages, tool_result)
        #print("Message after adding Tool result: ", messages)
            



In [8]:
messages = []
messages.append(
    {
        "role": "user",
        "content": "Set a reminder for 20 days from now to take medication"
    }
)
print("Prompt:", messages)
response = run_conversation(messages)

print("Final Response: ", response.content)

Prompt: [{'role': 'user', 'content': 'Set a reminder for 20 days from now to take medication'}]
----
Setting the following reminder for 2026-06-11T19:08:22:
Take medication
----
Final Response:  [TextBlock(citations=None, text="Perfect! I've set a reminder for **June 11, 2026 at 7:08 PM** (20 days from now) to take medication. You'll receive a notification at that time.", type='text')]
